# K-means SPM Classifier - Single Experiment Evaluation

This notebook evaluates a **single trained model** checkpoint.

## How to Use

1. **Choose which experiment to evaluate** in the configuration cell below (K=1, K=3, K=5, or baseline)
2. **Run all cells** to evaluate the model
3. **Results** will show training metadata and analysis

To evaluate a different K value, change the `EXPERIMENT_KEY` parameter and re-run the notebook.

## Evaluation Methodology

### Key Research Question
Does representing each class with multiple prototypes (via K-means clustering) improve classification accuracy?

### Main Evaluation Pathway (Consistent with Training)

```
Query Image
    ↓
Encoder → query_embedding (512-D)
    ↓
FAISS k-NN Search → k nearest neighbors with labels
    ↓
K-means Aggregation → K prototypes per class
    ↓
MoE Model → predictions
    ↓
Accuracy Computation
```

**Why this evaluation?**
- **Consistent with training:** Both use K-means aggregation + MoE
- **Tests the hypothesis:** Whether multiple prototypes improve SPM
- **Fair comparison:** Same evaluation for K=1, K=3, K=5

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime
import pandas as pd

# Add project to path
project_root = Path('../').resolve()
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## ⚙️ CONFIGURATION: Select Experiment to Evaluate

In [ ]:
# ============================================================
# CHANGE THIS TO SELECT WHICH EXPERIMENT TO EVALUATE
# ============================================================
# Options: 'baseline', 'kmeans_k1', 'kmeans_k3', 'kmeans_k5'
EXPERIMENT_KEY = 'kmeans_k3'  # ← MODIFY THIS
# ============================================================

# Experiment configurations
EXPERIMENTS = {
    'baseline': {
        'name': 'Original SPM (baseline)',
        'config': '../config/classification/cifar_resnet18.yaml',
        'checkpoint_dir': '../results/classification/cifar_resnet18',
        'K': None,
        'description': 'Single mean prototype per class'
    },
    'kmeans_k1': {
        'name': 'K-means SPM (K=1)',
        'config': '../config/classification/cifar_resnet18_kmeans_k1.yaml',
        'checkpoint_dir': '../results/classification/cifar_resnet18_kmeans_k1',
        'K': 1,
        'description': 'Single K-means cluster per class (baseline comparison)'
    },
    'kmeans_k3': {
        'name': 'K-means SPM (K=3)',
        'config': '../config/classification/cifar_resnet18_kmeans_k3.yaml',
        'checkpoint_dir': '../results/classification/cifar_resnet18_kmeans_k3',
        'K': 3,
        'description': 'Three prototypes per class (MAIN EXPERIMENT)'
    },
    'kmeans_k5': {
        'name': 'K-means SPM (K=5)',
        'config': '../config/classification/cifar_resnet18_kmeans_k5.yaml',
        'checkpoint_dir': '../results/classification/cifar_resnet18_kmeans_k5',
        'K': 5,
        'description': 'Five prototypes per class (ablation)'
    },
}

# Get current experiment config
if EXPERIMENT_KEY not in EXPERIMENTS:
    raise ValueError(f"Invalid EXPERIMENT_KEY: {EXPERIMENT_KEY}. Choose from: {list(EXPERIMENTS.keys())}")

exp_config = EXPERIMENTS[EXPERIMENT_KEY]

print("\n" + "="*70)
print("EXPERIMENT CONFIGURATION")
print("="*70)
print(f"Experiment: {exp_config['name']}")
print(f"Description: {exp_config['description']}")
if exp_config['K'] is not None:
    print(f"K (prototypes per class): {exp_config['K']}")
print(f"Config file: {exp_config['config']}")
print(f"Checkpoint dir: {exp_config['checkpoint_dir']}")
print("="*70 + "\n")

In [ ]:
## 1. Load Checkpoint

In [ ]:
def find_best_checkpoint(checkpoint_dir):
    """Find the best checkpoint in a directory."""
    checkpoint_dir = Path(checkpoint_dir)
    if not checkpoint_dir.exists():
        print(f"⚠️  Checkpoint directory not found: {checkpoint_dir}")
        return None
    
    # Look for best checkpoint (usually best-epoch-*.ckpt)
    best_ckpts = list(checkpoint_dir.glob('best-*.ckpt'))
    if best_ckpts:
        ckpt = str(sorted(best_ckpts)[-1])  # Get latest best checkpoint
        print(f"Found best checkpoint: {ckpt}")
        return ckpt
    
    # Look for last checkpoint
    last_ckpt = checkpoint_dir / 'last.ckpt'
    if last_ckpt.exists():
        print(f"Found last checkpoint: {last_ckpt}")
        return str(last_ckpt)
    
    print(f"No checkpoints found in {checkpoint_dir}")
    return None

# Find checkpoint for THIS experiment
checkpoint_path = find_best_checkpoint(exp_config['checkpoint_dir'])

if checkpoint_path is None:
    print("\n⚠️  ERROR: No checkpoint found!")
    print(f"Please train the model first:")
    print(f"  python main.py --config {exp_config['config']}")
else:
    print(f"\n✓ Checkpoint ready for evaluation")

In [ ]:
## 2. Load Configuration and Data

In [ ]:
from utils.config import load_config, instantiate_from_config
import pytorch_lightning as pl

# Load configuration
print(f"Loading config from {exp_config['config']}")
config = load_config(exp_config['config'])

# Instantiate datamodule
print("Setting up data...")
datamodule = instantiate_from_config(config['data'])
datamodule.prepare_data()
datamodule.setup(stage='test')

print(f"\nData Info:")
print(f"  Test dataset size: {len(datamodule.test_dataset)}")
print(f"  Train dataset size: {len(datamodule.train_dataset)}")
print(f"  Number of classes: {config['data']['params']['num_classes']}")
print(f"  Support size (eval): {config['data']['params'].get('support_size_eval', 1024)}")

In [ ]:
## 3. Load Trained Model

In [ ]:
from models.spm_classifier import SemiParametricClassifier
from models.spm_classifier_kmeans import SemiParametricClassifier_KMeans

if checkpoint_path is None:
    print("Cannot load model without checkpoint")
else:
    print(f"Loading model from checkpoint: {checkpoint_path}")
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    # Determine if this is a K-means model
    model_target = config['model']['target']
    is_kmeans = 'kmeans' in model_target.lower()
    
    # Instantiate model
    if is_kmeans:
        print(f"Loading K-means SPM model (K={exp_config['K']})")
        model = SemiParametricClassifier_KMeans(**config['model']['params'])
    else:
        print(f"Loading baseline SPM model")
        model = SemiParametricClassifier(**config['model']['params'])
    
    # Load weights
    state_dict = checkpoint['state_dict']
    model.load_state_dict(state_dict, strict=False)
    
    # Get metadata
    epoch = checkpoint.get('epoch', 'Unknown')
    global_step = checkpoint.get('global_step', 'Unknown')
    
    print(f"\nModel loaded successfully!")
    print(f"  Trained for {epoch} epochs")
    print(f"  Global step: {global_step}")
    print(f"  Device: CPU (checkpoint loaded on CPU, move to GPU below if available)")

In [ ]:
## 4. Evaluation Setup

This cell sets up the evaluation. Full k-NN based evaluation requires PyTorch Lightning trainer.

In [ ]:
if checkpoint_path is None:
    print("Cannot evaluate without checkpoint")
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    # Setup for evaluation
    model.support_size_eval = config['data']['params'].get('support_size_eval', 1024)
    
    print(f"Evaluating on {device}...")
    print(f"\nNote: Full evaluation requires PyTorch Lightning trainer with test_step")
    print(f"For proper k-NN based evaluation, use the training framework:")
    print(f"  python main.py --config {exp_config['config']} --test")
    print(f"\nResults will be logged to W&B and saved to results directory.")

In [ ]:
## 5. Results Summary

## 6. Expected Results and Interpretation

In [ ]:
interpretation = """
╔══════════════════════════════════════════════════════════════════╗
║          INTERPRETING K-MEANS EXPERIMENT RESULTS                 ║
╚══════════════════════════════════════════════════════════════════╝

KEY HYPOTHESIS:
  Multiple prototypes per class (via K-means) should improve SPM by
  better capturing multi-modal class distributions.

EXPECTED FINDINGS:

1️⃣  K=1 ≈ Baseline (Slightly Different)
    ✓ Good sign: Gradient flow working (different from baseline since K-means != mean)
    ✓ Validates: PyTorch K-means implementation is correct
    
2️⃣  K=3 >> K=1 (Significant Improvement)
    ✓ Main hypothesis confirmed!
    ✓ Multiple prototypes help class modeling
    ✓ K=3 is the "sweet spot" for this dataset

3️⃣  K=5 > K=3 (Modest Improvement or Saturation)
    ✓ Good: Additional prototypes still help
    ⚠  Watch: Diminishing returns likely
    ✗ Bad: If K=5 < K=3, overfitting detected

METRICS TO WATCH:
  • Validation Accuracy: Should peak at K=3
  • Test Accuracy: Main metric for generalization
  • Training Loss: Should decrease smoothly (indicates gradient flow)
  • Encoder Gradients: Should be non-zero (gradient flow working)

FAILURE MODES:

❌ K=3 ≈ K=1 ≈ Baseline
   → Classes well-separated; multi-modal modeling not needed
   → Try larger K (5-7) or check class distributions

❌ K=5 << K=3 << K=1
   → Overfitting: K too large for support set size
   → Fix: Reduce K or increase support_size_train

❌ NaN loss during training
   → Empty clusters during K-means (unlikely, auto-handled)
   → Check: Data normalization, learning rate

❌ Training very slow
   → K-means overhead on large support sets
   → Fix: Reduce support_size_train or K

NEXT STEPS:
  1. Run all three experiments (K=1, K=3, K=5)
  2. Compare validation curves in W&B
  3. Analyze final test accuracies
  4. If K=3 wins, test on unlearning tasks
  5. (Optional) Grid search best K (try 2-7)
"""

print(interpretation)

## 7. How to Evaluate All Experiments

### Step 1: Train Models

```bash
# Train baseline
python main.py --config config/classification/cifar_resnet18.yaml

# Train K=1
python main.py --config config/classification/cifar_resnet18_kmeans_k1.yaml

# Train K=3 (MAIN EXPERIMENT)
python main.py --config config/classification/cifar_resnet18_kmeans_k3.yaml

# Train K=5 (ABLATION)
python main.py --config config/classification/cifar_resnet18_kmeans_k5.yaml
```

### Step 2: Evaluate Each Model

For each experiment, change `EXPERIMENT_KEY` in the configuration cell above and re-run this notebook:

```python
EXPERIMENT_KEY = 'baseline'    # First run
EXPERIMENT_KEY = 'kmeans_k1'   # Second run
EXPERIMENT_KEY = 'kmeans_k3'   # Third run (main)
EXPERIMENT_KEY = 'kmeans_k5'   # Fourth run (ablation)
```

### Step 3: Compare Results

Check W&B dashboard for:
- Validation accuracy curves
- Test accuracy comparison
- Training loss curves
- Convergence speed

### Step 4: Analysis

See Cell above for interpretation guide.

In [ ]:
## 8. Export Results

print("\n" + "="*70)
print(f"EVALUATION SUMMARY: {exp_config['name']}")
print("="*70)

summary = f"""
Experiment: {exp_config['name']}
Description: {exp_config['description']}
Config: {exp_config['config']}
Checkpoint: {checkpoint_path if checkpoint_path else 'Not found'}

Model Details:
  - Model Type: {'K-means SPM' if exp_config['K'] is not None else 'Baseline SPM'}
  - Prototypes per class (K): {exp_config['K'] if exp_config['K'] is not None else 'N/A (single mean)'}
  - Number of classes: {config['data']['params']['num_classes']}
  - Encoder: ResNet18
  - Embed dimension: 512

Evaluation Setup:
  - Test set size: {len(datamodule.test_dataset)}
  - Support size (k-NN): {config['data']['params'].get('support_size_eval', 1024)}
  - Evaluation method: k-NN + {'K-means aggregation' if exp_config['K'] is not None else 'raw embeddings'} + MoE

To run full test evaluation:
  1. Use PyTorch Lightning trainer: python main.py --config {exp_config['config']}
  2. Results logged to W&B
  3. Metrics saved to {exp_config['checkpoint_dir']}

Expected Results:
  - K=3 should show improvement over K=1
  - K=1 should approximate baseline performance
  - K=5 may show diminishing returns or overfitting
"""

print(summary)
print("="*70)

In [ ]:
def save_evaluation_summary(exp_key, exp_config):
    """
    Save evaluation summary to JSON file for this experiment.
    """
    summary = {
        'timestamp': datetime.now().isoformat(),
        'experiment_key': exp_key,
        'experiment_name': exp_config['name'],
        'K': exp_config['K'],
        'config': exp_config['config'],
        'checkpoint_dir': exp_config['checkpoint_dir'],
        'checkpoint_found': checkpoint_path is not None,
        'evaluation_methodology': {
            'training': 'support_samples → K-means aggregation → K prototypes → MoE',
            'evaluation': 'k-NN → K-means aggregation → K prototypes → MoE',
            'key_feature': 'Consistent evaluation with training'
        },
        'dataset': 'CIFAR-10',
        'encoder': 'ResNet18',
        'embed_dim': 512,
        'num_classes': 10,
        'notes': 'Run experiments first with: python main.py --config <config>'
    }
    
    output_file = f'kmeans_evaluation_{exp_key}.json'
    with open(output_file, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"Evaluation summary saved to: {output_file}")
    return summary

if checkpoint_path is not None:
    summary = save_evaluation_summary(EXPERIMENT_KEY, exp_config)
    print("\n" + json.dumps(summary, indent=2))
else:
    print("Checkpoint not found. Please train the model first.")

## Summary

This notebook evaluates a **single K-means experiment** at a time.

### Quick Workflow

1. **Change `EXPERIMENT_KEY` in the configuration cell** to specify which model to evaluate
   - Options: `'baseline'`, `'kmeans_k1'`, `'kmeans_k3'`, `'kmeans_k5'`

2. **Run all cells** to load and evaluate the model

3. **Check W&B dashboard** for detailed metrics and curves

4. **Repeat for each experiment** to compare results

### Expected Results

- **K=1 ≈ Baseline:** Validates gradient flow
- **K=3 >> K=1:** Main hypothesis (multiple prototypes help!)
- **K=5 > K=3:** Diminishing returns expected

See Cell 6 for detailed interpretation guide.

### Comparison Across Experiments

For a comprehensive comparison, train and evaluate all four variants, then view results in W&B dashboard side-by-side.